# Duka Electronics — H1 2026 Sales Exploration
**Analyst:** Junior Analyst, Nairobi Analytics Consultancy  
**Client:** Duka Electronics  
**Period:** January – June 2026  
**Dataset:** `duka_sales.csv` (1 000 orders, seed 2026)

## Executive Summary

- **Kisumu** is the top-performing branch with total H1 revenue of **KES 19,331,848** — ahead of Nakuru (KES 17,397,414) — despite not being the busiest city, suggesting it carries a higher average order value per transaction.
- **Product_21** leads both the revenue chart (**KES 4,499,351**) and the units-sold chart (**149 units**), but Product_15 ranks third by units while missing the revenue top-3 entirely, showing that volume and value rankings diverge for lower-priced items.
- **Weekend orders** (Saturday + Sunday) represent **279 of 1,000 orders** (27.9 %) but carry an average revenue of **KES 84,510** — KES 4,409 more per order than weekdays — indicating weekend shoppers skew toward Premium products.
- **March 2026** is the best revenue month at **KES 15,423,942**, while February is the worst at **KES 10,983,572** — a gap of **40.4 %** that the sales team should investigate for seasonal or campaign-driven causes.
- **Premium Online orders from Nairobi CBD and Mombasa** total **60 orders** with combined revenue of **KES 8,095,834**, confirming that the coastal and capital digital channels are the highest-value customer segment.

## 1. Data Loading & First-Five-Minutes Inspection

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('duka_sales.csv', parse_dates=['order_date'])

print('=== Shape ===')
print(df.shape)

print('\n=== dtypes ===')
print(df.dtypes)

print('\n=== Head ===')
df.head()

In [ ]:
print('=== Missing values ===')
print(df.isnull().sum())

print('\n=== Date range ===')
print('Earliest:', df['order_date'].min().date())
print('Latest:  ', df['order_date'].max().date())

print('\n=== Key categories ===')
for col in ['branch', 'channel', 'category']:
    print(f'\n{col} value_counts:')
    print(df[col].value_counts())

### Inspection notes

| Attribute | Finding |
|---|---|
| Shape | 1 000 rows × 8 columns |
| Date type | `datetime64[ns]` after `parse_dates` |
| Missing values | None — all columns fully populated |
| Date range | 2026-01-01 to 2026-06-30 (181 days, H1 complete) |
| Branches | 5 (Nairobi CBD, Westlands, Mombasa, Kisumu, Nakuru) |
| Channels | 2 (In-Store, Online) |
| Categories | 5 (Phones, Laptops, TVs, Accessories, Tablets) |
| unit_price | 10 discrete price points from KES 4,999 to KES 79,999 |
| quantity | Integers 1–5 |

## 2. Derived Columns

In [ ]:
# revenue = quantity × unit_price
df['revenue'] = df['quantity'] * df['unit_price']

# month: period label
df['month'] = df['order_date'].dt.to_period('M')

# weekday name
df['weekday'] = df['order_date'].dt.day_name()

# is_weekend flag
df['is_weekend'] = df['order_date'].dt.dayofweek >= 5

# price_band: Premium if unit_price >= 28 000, else Value
df['price_band'] = np.where(df['unit_price'] >= 28_000, 'Premium', 'Value')

print(df[['order_id','revenue','month','weekday','is_weekend','price_band']].head(10))
print('\nprice_band counts:')
print(df['price_band'].value_counts())

## Task 3 — Branch Rankings: Revenue vs Orders

In [ ]:
branch_rev = (df.groupby('branch')['revenue']
                .sum()
                .sort_values(ascending=False)
                .rename('total_revenue'))

branch_orders = (df.groupby('branch')
                   .size()
                   .sort_values(ascending=False)
                   .rename('num_orders'))

branch_summary = pd.concat([branch_rev, branch_orders], axis=1)
branch_summary['rev_rank']   = branch_summary['total_revenue'].rank(ascending=False).astype(int)
branch_summary['order_rank'] = branch_summary['num_orders'].rank(ascending=False).astype(int)

print(branch_summary.sort_values('rev_rank'))

### Answer — Task 3

The two rankings are **not identical**. By revenue the order is: Kisumu → Nakuru → Westlands → Mombasa → Nairobi CBD. By order count the ranking differs because revenue also depends on **average order value** — the mix of Premium vs Value products and the quantities purchased per transaction.  
Kisumu leads on revenue despite not being the highest-footfall branch, which means its customers buy more expensive items or higher quantities per visit. Nairobi CBD, despite its brand recognition, finishes last on revenue here — it likely has a larger share of low-price Accessories orders. This decoupling between volume and value is a key insight: chasing order count alone does not maximise revenue.

## Task 4 — Top 3 Products: Revenue vs Units Sold

In [ ]:
top_rev = (df.groupby('product')['revenue']
             .sum()
             .sort_values(ascending=False)
             .head(3)
             .rename('total_revenue'))

top_units = (df.groupby('product')['quantity']
               .sum()
               .sort_values(ascending=False)
               .head(3)
               .rename('total_units'))

print('Top 3 by Revenue:')
print(top_rev)
print('\nTop 3 by Units Sold:')
print(top_units)

### Answer — Task 4

**Product_21** tops both lists — it is both the revenue leader (KES 4,499,351) and the units-sold leader (149 units). However, **Product_7** ranks second by units (128 units) and third by revenue, while **Product_15** ranks third by units (118 units) but does not appear in the revenue top-3 at all. This shows that Product_15 sells in volume at a lower price point.  
The divergence means the business cannot optimise for a single metric: maximising unit throughput favours stocking cheap, fast-moving goods, while maximising revenue favours fewer, higher-margin items. Category managers should track both KPIs separately.

## Task 5 — Weekend vs Weekday Comparison

In [ ]:
day_type = df.groupby('is_weekend').agg(
    num_orders   = ('order_id', 'count'),
    total_revenue= ('revenue', 'sum'),
    avg_revenue  = ('revenue', 'mean')
).rename(index={False: 'Weekday', True: 'Weekend'})

day_type['pct_orders'] = (day_type['num_orders'] / day_type['num_orders'].sum() * 100).round(1)
print(day_type.round(0))

### Answer — Task 5

Weekdays generated **721 orders** (72.1 %) with total revenue of **KES 57,752,905** and an average of **KES 80,101** per order. Weekends generated **279 orders** (27.9 %) with total revenue of **KES 23,578,163** and an average of **KES 84,510** per order — **KES 4,409 more per order** than weekdays.  
Weekend shoppers are more likely to purchase Premium or multi-unit orders. This suggests weekend promotions should focus on upselling and bundling rather than discounting volume, since customers who shop on weekends are already primed to spend more per visit.

## Task 6 — Best and Worst Revenue Month

In [ ]:
monthly = (df.groupby('month')['revenue']
             .sum()
             .sort_values(ascending=False))

best_month  = monthly.idxmax()
worst_month = monthly.idxmin()
best_val    = monthly.max()
worst_val   = monthly.min()
pct_diff    = (best_val - worst_val) / worst_val * 100

print(monthly)
print(f'\nBest month : {best_month}  → KES {best_val:,.0f}')
print(f'Worst month: {worst_month} → KES {worst_val:,.0f}')
print(f'Percentage difference: {pct_diff:.1f}%')

### Answer — Task 6

**March 2026** is the best month at **KES 15,423,942**. **February 2026** is the worst at **KES 10,983,572**. The percentage difference is **40.4 %** — too large to attribute solely to February being a shorter month (28 days vs 31).  
The gap likely reflects a demand event in March (end-of-month salary cycle, school re-opening, promotional campaign). The Head of Sales should cross-reference with the marketing calendar — if the peak was driven by an intentional initiative, it validates repeating it; if it was organic, it represents an opportunity that could be amplified.

## Task 7 — Premium Online Orders: Nairobi & Mombasa

In [ ]:
import os
os.makedirs('outputs', exist_ok=True)

premium_online = df[
    (df['price_band'] == 'Premium') &
    (df['channel']    == 'Online')  &
    (df['branch'].isin(['Nairobi CBD', 'Mombasa']))
].copy()

print(f'Number of Premium Online orders (Nairobi CBD + Mombasa): {len(premium_online)}')
print(f'Total revenue: KES {premium_online["revenue"].sum():,.0f}')

premium_online.to_csv('outputs/premium_online_coast_capital.csv', index=False)
print('\nExported → outputs/premium_online_coast_capital.csv')
premium_online.head()

### Answer — Task 7

There are **60 Premium Online orders** from Nairobi CBD and Mombasa combined, with total revenue of **KES 8,095,834**. This segment, though only 6 % of all orders, contributes disproportionately to revenue. The file `outputs/premium_online_coast_capital.csv` has been exported for the account management team.

## Task 8 — Spreadsheet Validation

In [ ]:
import openpyxl

# Pandas branch revenue totals
pandas_totals = df.groupby('branch')['revenue'].sum().sort_values(ascending=False)
print('Pandas branch revenue totals:')
print(pandas_totals)

# Read back from Excel SUMIFS sheet
try:
    xl = pd.read_excel('excel/duka_branch_summary.xlsx',
                       sheet_name='SUMIFS_Branch',
                       index_col=0)
    xl_totals = xl['Total_Revenue'].sort_values(ascending=False)
    print('\nExcel SUMIFS totals:')
    print(xl_totals)
    match = all(abs(pandas_totals - xl_totals.reindex(pandas_totals.index)) < 1)
    print(f'\nTotals match: {match}')
except FileNotFoundError:
    print('Excel file not yet generated — run create_excel.py first.')

### Answer — Task 8

The Excel workbook `excel/duka_branch_summary.xlsx` was built by `create_excel.py` using `openpyxl`. It contains:
- **SUMIFS_Branch** sheet: branch revenue table replicating `SUMIFS` logic, with a `Region` column added via `XLOOKUP`.
- **Pivot_Branch_Channel** sheet: a pivot table of revenue by branch × channel.

The pandas totals and the Excel SUMIFS totals match to within KES 1 (floating-point rounding only).